# PROJECT 2 : FIRST NEW VERSION OF SPR FUNCTION
This consist of using OLS + VAR (1) to improove the results of the interpolation function

## Import libraries

In [1]:
import numpy as np           # For numerical computing
import pandas as pd          # For data manipulation and analysis
import matplotlib.pyplot as plt  # For plotting and visualization
import scipy                 # For scientific computing
import sklearn               # For machine learning
import geopandas as gpd      # For geospatial data
import rasterio              # For raster data processing
import xarray                # For working with labeled multi-dimensional arrays
import tensorflow as tf      # For deep learning and neural networks
import torch                 # For deep learning with PyTorch
import seaborn as sns        # For statistical data visualization
import plotly.express as px  # For interactive plots
import statsmodels.api as sm  # For statistical modeling
import folium               # For interactive maps
import networkx as nx        # For complex networks
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import random
import os
from sklearn.utils import resample
import joblib  # for saving data
from multiprocessing import Pool
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
import multiprocessing
from joblib import Parallel, delayed
from tqdm import tqdm
from pykrige.ok import OrdinaryKriging
from skgstat import Variogram
from pykrige.uk import UniversalKriging
from pykrige.rk import Krige
from shapely.geometry import Point
import geopandas as gpd
import warnings
from pyproj import CRS, Transformer
from pandas.plotting import autocorrelation_plot
from statsmodels.graphics.tsaplots import plot_acf

2026-04-24 16:09:26.818680: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-24 16:09:26.842687: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-24 16:09:27.330545: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


## Auto saving my script every 60 seconds

In [2]:
%autosave 60

Autosaving every 60 seconds


In [3]:
70*20*4

5600

# Loading data

In [4]:
data_coord = pd.read_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data/data_coord.csv")
data_pr = pd.read_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data/data_pr.csv")
data_tasmin = pd.read_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data/data_tasmin.csv")
data_tasmax = pd.read_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data/data_tasmax.csv")
# clim_pr = pd.read_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data/clim_pr.csv")
# clim_tasmin = pd.read_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data/clim_tasmin.csv")
# clim_tasmax = pd.read_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data/clim_tasmax.csv")

# Defining period and indexes

In [5]:
# Extract longitude and latitude columns
rlon = data_coord["lon"]
rlat = data_coord["lat"]

# Define region A
region_A1 = data_coord[(rlat >= -2) & (rlat <= 4) & (rlon >= 13) & (rlon <= 19)].index
region_A2 = data_coord[(rlat >= -1) & (rlat <= 3) & (rlon >= 14) & (rlon <= 18)].index
region_A3 = data_coord[(rlat >= 0) & (rlat <= 2) & (rlon >= 15) & (rlon <= 17)].index

# Define region B
region_B1 = data_coord[(rlat >= 7) & (rlat <= 13) & (rlon >= 14) & (rlon <= 20)].index
region_B2 = data_coord[(rlat >= 8) & (rlat <= 12) & (rlon >= 15) & (rlon <= 19)].index
region_B3 = data_coord[(rlat >= 9) & (rlat <= 11) & (rlon >= 16) & (rlon <= 18)].index

# Defining the periods (1-based indexing in R becomes 0-based in Python)
rcm_period = np.arange(0, 10950)         # 0 to 10949 → 30 years
obs_period_1 = np.arange(7300, 10950)    # 7301:10950 → 7300 to 10949
obs_period_2 = np.arange(10950, 14600)   # 10951:14600 → 10950 to 14599
obs_period_3 = np.arange(14600, 18250)   # 14601:18250 → 14600 to 18249

# Periode de validation et de test
obs_period_hp = np.arange(7300, 9125)
obs_period_test = np.arange(9125, 10950)

## Choose the period and the region
obs_period = obs_period_test # choose the period
region = region_A1 # choose the region
n_years_obs = 5 # number of years in the observation period
n_years_ano = 30 # number of years in the anomaly period

## Define the seed : Set the random seed for reproducibility
my_seed = 2244677 # seed for reproducibility

# Data preparation : defining data on the period and region considered

In [6]:
# Assuming data.pr, data.tasmin, data.tasmax are numpy arrays or pandas DataFrames
# Define these with actual data or load from files
# Example placeholders (replace these with real data):
# data_pr, data_tasmin, data_tasmax = ...

# Example: data should be NumPy arrays or similar
# Using .loc because region_colnames are column names
# rcm_period and obs_period are row positions and region_colnames are column names
## RCM data
region_colnames = [f"V{col}" for col in region]
data_pr_grid = data_pr.loc[rcm_period, region_colnames]
data_pr_grid[data_pr_grid < 0.5] = 0  # remove noise
data_tasmin_grid = data_tasmin.loc[rcm_period, region_colnames]
data_tasmax_grid = data_tasmax.loc[rcm_period, region_colnames]


## Observed data
data_pr_obs = data_pr.loc[obs_period, region_colnames]
# data_pr_obs[data_pr_obs < 0.5] = 0  # remove noise
data_pr_obs[data_pr_obs <= 1e-5] = 1e-5  # lower bound
data_pr_obs = np.log(np.exp(data_pr_obs) - 1)  # transform for positivity
data_tasmin_obs = data_tasmin.loc[obs_period, region_colnames]
data_tasmax_obs = data_tasmax.loc[obs_period, region_colnames]

## Coordinate data for KED interpolation
data_coord_grid = data_coord.loc[region, :]

## Original data for comparison
data_pr_obs_orig = data_pr.loc[obs_period, region_colnames]
data_pr_obs_orig[data_pr_obs_orig < 0.5] = 0  # remove noise
data_tasmin_obs_orig = data_tasmin.loc[obs_period, region_colnames]
data_tasmax_obs_orig = data_tasmax.loc[obs_period, region_colnames]

## Interpolation with 99.9% NA : Densité de 0.1%

# Train and test set

In [7]:
# --- Custom rounding function (round .5 up) ---
perc = 0.999  # % of columns to set as test
def custom_round(x):
    return int(np.ceil(x) if x - int(x) == 0.5 else round(x))

# Select test indexes
np.random.seed(my_seed)
random.seed(my_seed)
n_days, n_cols = data_pr_obs.shape
n_test = custom_round(perc * n_cols)
test_indexes = np.random.choice(n_cols, size=n_test, replace=False)
test_indexes.sort()  # Sort for easier reading
train_indexes = [i for i in range(n_cols) if i not in test_indexes]

# --- Set test indices to NaN in all datasets ---
data_pr_obs.iloc[:, test_indexes] = np.nan
data_tasmin_obs.iloc[:, test_indexes] = np.nan
data_tasmax_obs.iloc[:, test_indexes] = np.nan

## Convert all data to nympy

In [8]:
# Convert all data to numpy
## Precipitation
data_pr_obs_np = data_pr_obs.values if isinstance(data_pr_obs, pd.DataFrame) else data_pr_obs
data_pr_obs_orig_np = data_pr_obs_orig.values if isinstance(data_pr_obs_orig, pd.DataFrame) else data_pr_obs_orig

## Tmin
data_tasmin_obs_np = data_tasmin_obs.values if isinstance(data_tasmin_obs, pd.DataFrame) else data_tasmin_obs
data_tasmin_obs_orig_np = data_tasmin_obs_orig.values if isinstance(data_tasmin_obs_orig, pd.DataFrame) else data_tasmin_obs_orig

## Tmax
data_tasmax_obs_np = data_tasmax_obs.values if isinstance(data_tasmax_obs, pd.DataFrame) else data_tasmax_obs
data_tasmax_obs_orig_np = data_tasmax_obs_orig.values if isinstance(data_tasmax_obs_orig, pd.DataFrame) else data_tasmax_obs_orig

# ## Original data
# data_pr_obs_orig_np = data_pr_obs_orig.values if isinstance(data_pr_obs_orig, pd.DataFrame) else data_pr_obs_orig
# data_tasmin_obs_orig_np = data_tasmin_obs_orig.values if isinstance(data_tasmin_obs_orig, pd.DataFrame) else data_tasmin_obs_orig
# data_tasmax_obs_orig_np = data_tasmax_obs_orig.values if isinstance(data_tasmax_obs_orig, pd.DataFrame) els

# Interpolation with OLS

In [9]:
# Interpolation functions with OLS : reference model
# SPR original function
def spr_interp(data_rcm_grid, data_obs_grid, n_spatterns=10, n_jobs=-1):
    """
    Performs Spatial Pattern Regression (SPR) interpolation using PCA-based spatial patterns
    derived from RCM data to interpolate missing observation data, in parallel over days.

    Parameters:
    - data_rcm_grid: np.ndarray of shape (n_days, n_stations)
        Grid of RCM data used to extract spatial patterns.
    - data_obs_grid: np.ndarray of shape (m_days, n_stations)
        Grid of observation data (with missing values, NaNs).
    - n_spatterns: int, default=10
        Number of spatial patterns (principal components) to retain and use.
    - n_jobs: int, default=-1
        Number of parallel jobs. Use -1 to use all available cores.

    Returns:
    - data_interp_spr: np.ndarray of shape (m_days, n_stations)
        Interpolated observation data with missing values filled in.
    """

    # Convert inputs to numpy arrays if they are pandas DataFrames
    data_rcm_grid = data_rcm_grid.values if isinstance(data_rcm_grid, pd.DataFrame) else data_rcm_grid
    data_obs_grid = data_obs_grid.values if isinstance(data_obs_grid, pd.DataFrame) else data_obs_grid

    # Step 0: Input validation
    if data_rcm_grid.shape[1] != data_obs_grid.shape[1]:
        raise ValueError("RCM and observation grids must have the same number of stations (columns).")
    if n_spatterns > data_rcm_grid.shape[1]:
        raise ValueError("Number of spatial patterns cannot exceed the number of stations.")

    # Step 1: Standardize RCM data (mean-center only)
    scaler = StandardScaler(with_mean=True, with_std=False)
    data_rcm_scaled = scaler.fit_transform(data_rcm_grid)  # Center each column (station)
    centres_pca = scaler.mean_  # Save the means for re-centering later

    # Step 2: PCA on RCM data to extract spatial patterns
    svd_rcm = TruncatedSVD(n_components=n_spatterns, random_state=0)
    svd_rcm.fit(data_rcm_scaled)
    spatterns = svd_rcm.components_.T  # Shape: (n_stations, n_spatterns)

    # Step 3: Identify and remove stations with NaNs on first day
    first_day = data_obs_grid[0, :]  # First row of observation data
    indexes_NA_col = np.where(np.isnan(first_day))[0]  # Indexes of columns with NaNs
    data_obs_valid = np.delete(data_obs_grid, indexes_NA_col, axis=1)  # Remove those columns
    spatterns_model = np.delete(spatterns, indexes_NA_col, axis=0)      # Remove corresponding rows
    centres_model = np.delete(centres_pca, indexes_NA_col)              # Remove corresponding means

    nrow_obs, ncol_obs = data_obs_grid.shape

    # Step 4: Define interpolation function for one day
    def interpolate_one_day(i):
        # Get and center current day's valid observation values (those not removed)
        obs_day = data_obs_valid[i, :]
        obs_day_centered = obs_day - centres_model

        # Fit the linear regression model to find pattern weights for each day
        # Note: We use fit_intercept=False because we already centered the data
        model = LinearRegression(fit_intercept=False)
        model.fit(spatterns_model, obs_day_centered)
        pattern_weights = model.coef_.reshape(-1, 1)  # Column vector (n_spatterns, 1)

        # Reconstruct the full spatial field using all spatial patterns
        # Note: We use the original spatterns (not spatterns_model) to reconstruct
        # the full spatial field, as we want to fill in the missing values
        # across all stations, not just the valid ones.
        reconstructed_day = spatterns @ pattern_weights + centres_pca.reshape(-1, 1)
        return reconstructed_day.ravel(), pattern_weights.ravel()  # Return as 1D arrays

    # Step 5: Run interpolation in parallel
    interpolated_days = Parallel(n_jobs=n_jobs)(
        delayed(interpolate_one_day)(i) for i in tqdm(range(nrow_obs), desc="Interpolating days")
    )

    # Step 6: Stack all daily results into final array (n_days * n_stations)
    # Note: Each interpolated day is a 1D array, we need to stack them into a 2D array
    # Shape: (n_days, n_stations)
    # Step 6: Separate reconstructions and pattern weights
    interpolated_days, pattern_weights_list = zip(*interpolated_days)
    data_interp_spr = np.vstack(interpolated_days)          # (n_days, n_stations)
    all_pattern_weights = np.vstack(pattern_weights_list)   # (n_days, n_spatterns)

    return data_interp_spr#, all_pattern_weights

## PRECIP

In [10]:
# Convert pd.DataFrame to np.ndarray with .values cauze spr_interp process NUMPY arrays.
data_pr_grid_np = data_pr_grid.values if isinstance(data_pr_grid, pd.DataFrame) else data_pr_grid
data_pr_obs_np = data_pr_obs.values if isinstance(data_pr_obs, pd.DataFrame) else data_pr_obs

# Remove test columns from data_pr_obs
data_pr_test = np.delete(data_pr_obs_np, test_indexes, axis=1)

# Compute n_pr
n_pr = min(data_pr_grid.shape[1], data_pr_test.shape[1])

# Define proportion of available spatial patterns to use for the regression as predictors
prop_NA_pr = 0.67

# Compute number of spatial patterns
n_spatterns_pr = custom_round(n_pr * prop_NA_pr)

def apply_spr_with_postprocess(data_rcm_grid, data_obs_grid, n_spatterns):
    interp = spr_interp(data_rcm_grid, data_obs_grid, n_spatterns)

    # Transformation inverse pour garantir la positivité
    interp_transformed = np.log1p(np.exp(interp))  # log(1 + exp(x))

    # Seuil pour supprimer le bruit
    interp_transformed[interp_transformed < 0.5] = 0

    # Return the transformed interpolated data and pattern weights
    return interp_transformed

# Apply the function to the data and save the results
spr_ols_pr_A1_01 = apply_spr_with_postprocess(data_pr_grid_np, data_pr_obs_np, n_spatterns_pr)
spr_ols_pr_A1_01 = pd.DataFrame(spr_ols_pr_A1_01, columns=data_pr_obs.columns, index=data_pr_obs.index)
spr_ols_pr_A1_01.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Interpolation/Interp_A1/Simulations/spr_ols_pr_A1_01.csv")

Interpolating days: 100%|██████████| 1825/1825 [00:01<00:00, 1753.11it/s]


## Minimum temperature interpolation with SPR_ols

In [11]:
# Remove test columns from anomaly_tasmin_obs
data_tasmin_train = np.delete(data_tasmin_obs_np, test_indexes, axis=1)

# Compute n_tasmin
n_tasmin = min(data_tasmin_grid.shape[1], data_tasmin_train.shape[1])

# Define proportion of available spatial patterns to use for the regression as predictors
prop_NA_tasmin = 0.67

# Compute number of spatial patterns
n_spatterns_tasmin = custom_round(n_tasmin * prop_NA_tasmin)

# Interpolation with choosen hyperparameters for tmin
spr_ols_tmin_A1_01 = spr_interp(
    data_rcm_grid=data_tasmin_grid,
    data_obs_grid=data_tasmin_obs_np,
    n_spatterns=n_spatterns_tasmin
)

# # Save results as pandas DataFrame and CSF files
spr_ols_tmin_A1_01 = pd.DataFrame(spr_ols_tmin_A1_01, columns=data_tasmin_obs.columns, index=data_tasmin_obs.index)
spr_ols_tmin_A1_01.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Interpolation/Interp_A1/Simulations/spr_ols_tmin_A1_01.csv")
# beta_ols_tmin_A1_01 = pd.DataFrame(beta_ols_tmin_A1_01)
# beta_ols_tmin_A1_01.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data_interp/SPR_ols_10/beta_ols_tmin_A1_01.csv")
# anomaly_ols_tmin_A1_01 = pd.DataFrame(anomaly_ols_tmin_A1_01, columns=data_tasmin_obs.columns, index=data_tasmin_obs.index)
# anomaly_ols_tmin_A1_01.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data_interp/SPR_ols_10/anomaly_ols_tmin_A1_01.csv")

Interpolating days: 100%|██████████| 1825/1825 [00:00<00:00, 17847.15it/s]


## Maximum temperature interpolation with SPR_ols

In [12]:
# Remove test columns from anomaly_tasmax_obs
data_tasmax_train = np.delete(data_tasmax_obs_np, test_indexes, axis=1)

# Compute n_tasmax
n_tasmax = min(data_tasmax_grid.shape[1], data_tasmax_train.shape[1])

# Define proportion of available spatial patterns to use for the regression as predictors
prop_NA_tasmax = 0.67

# Compute number of spatial patterns
n_spatterns_tasmax = custom_round(n_tasmax * prop_NA_tasmax)

# Interpolation with choosen hyperparameters for tmax
spr_ols_tmax_A1_01 = spr_interp( 
    data_rcm_grid=data_tasmax_grid,
    data_obs_grid=data_tasmax_obs_np,
    n_spatterns=n_spatterns_tasmax
)

# # Save results as pandas DataFrame and CSF files
spr_ols_tmax_A1_01 = pd.DataFrame(spr_ols_tmax_A1_01, columns=data_tasmax_obs.columns, index=data_tasmax_obs.index)
spr_ols_tmax_A1_01.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Interpolation/Interp_A1/Simulations/spr_ols_tmax_A1_01.csv")
# beta_ols_tmax_A1_01 = pd.DataFrame(beta_ols_tmax_A1_01)
# beta_ols_tmax_A1_01.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data_interp/SPR_ols_10/beta_ols_tmax_A1_01.csv")
# anomaly_ols_tmax_A1_01 = pd.DataFrame(anomaly_ols_tmax_A1_01, columns=data_tasmax_obs.columns, index=data_tasmax_obs.index)
# anomaly_ols_tmax_A1_01.to_csv("/home/vihoua@labos.polymtl.ca/Projet_2/Data_interp/SPR_ols_10/anomaly_ols_tmax_A1_01.csv")

Interpolating days: 100%|██████████| 1825/1825 [00:00<00:00, 14012.80it/s]
